# 01 — Merge Datasets

Join survey + demographics + state-level coal share + plant proximity.
Print shape and `.head()` **before and after every merge** as a diagnostic check.

| | |
|---|---|
| **Inputs** | `data/raw/` (all four files) |
| **Outputs** | `data/processed/merged_analysis.csv` |

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_RAW  = Path("../data/raw")
DATA_PROC = Path("../data/processed")
DATA_PROC.mkdir(exist_ok=True)

# ── Helper: merge with diagnostic output ──────────────────────────────────────
def merge_check(left, right, on, how="left", label=""):
    """Merge two dataframes and print shape before/after + head."""
    print(f"\n{'='*55}")
    print(f"MERGE: {label}")
    print(f"  left:   {left.shape}")
    print(f"  right:  {right.shape}")
    merged = left.merge(right, on=on, how=how)
    print(f"  result: {merged.shape}")
    lost = left.shape[0] - merged.shape[0]
    if lost > 0:
        print(f"  WARNING: {lost} rows lost in merge — check join key: {on}")
    print(merged.head(2).to_string())
    return merged

In [ ]:
# ── 1. Load survey ────────────────────────────────────────────────────────────
survey = pd.read_excel(DATA_RAW / "renewables.xlsx")
print(f"Survey loaded: {survey.shape}")

# Rename DV for clarity
survey = survey.rename(columns={"Q137": "re_support"})

# Drop rows missing on DV
survey = survey.dropna(subset=["re_support"])
print(f"After dropping missing DV: {survey.shape}")

In [ ]:
# ── 2. Load demographics and merge onto survey ────────────────────────────────
demo = pd.read_csv(DATA_RAW / "spectrum.csv")
print(f"Demographics loaded: {demo.shape}")

# ⚠️  Adjust 'respondent_id' below to match the actual join key in your files
df = merge_check(survey, demo, on="respondent_id", how="left",
                 label="survey + demographics")

In [ ]:
# ── 3. Build coal share from EIA SEDS ────────────────────────────────────────
coal_raw = pd.read_excel(DATA_RAW / "Prod_dataset.xlsx")
print(f"Coal production loaded: {coal_raw.shape}")
print(f"Columns: {list(coal_raw.columns)}")

# ⚠️  Adjust column names below to match your actual file
# Expected columns: StateCode (or State), Year, CoalProduction, TotalProduction
# Filter to 1990–2023 for economic legacy measure
coal_filtered = coal_raw[coal_raw["Year"] >= 1990].copy()

# Compute coal share = coal / total energy production per state-year
coal_filtered["coal_share_year"] = (
    coal_filtered["CoalProduction"] / coal_filtered["TotalProduction"]
)

# Average across years per state
coal_share = (
    coal_filtered
    .groupby("StateCode")["coal_share_year"]
    .mean()
    .reset_index()
    .rename(columns={"StateCode": "state", "coal_share_year": "coal_share"})
)
print(f"\nCoal share by state: {coal_share.shape}")
print(coal_share.sort_values("coal_share", ascending=False).head(10))

In [ ]:
# ── 4. Merge coal share onto respondent data ──────────────────────────────────
# ⚠️  Adjust 'state' to match the state column name in your survey
df = merge_check(df, coal_share, on="state", how="left",
                 label="+ coal share")

print(f"\nMissing coal_share after merge: {df['coal_share'].isna().sum()}")

In [ ]:
# ── 5. Compute plant proximity ────────────────────────────────────────────────
plants = pd.read_excel(DATA_RAW / "plant.xlsx")
print(f"Plant data loaded: {plants.shape}")
print(f"Columns: {list(plants.columns)}")

# Filter to active coal and renewable plants
# ⚠️  Adjust 'Plant_Type' and 'Status' to match your actual column names
coal_plants = plants[plants["Plant_Type"].str.contains("Coal", case=False, na=False)]
re_plants   = plants[plants["Plant_Type"].str.contains("Solar|Wind", case=False, na=False)]

print(f"\nActive coal plants:      {len(coal_plants)}")
print(f"Active renewable plants: {len(re_plants)}")

In [ ]:
# ── 6. State-level plant proximity (simplified) ───────────────────────────────
# For full respondent-level proximity, use haversine distance to nearest plant.
# Here we use state-level count as a simpler proximity proxy.

# ⚠️  Adjust 'State' to match your plant data column
coal_count = coal_plants.groupby("State").size().reset_index(name="n_coal_plants")
re_count   = re_plants.groupby("State").size().reset_index(name="n_re_plants")
plant_summary = coal_count.merge(re_count, on="State", how="outer").fillna(0)
plant_summary = plant_summary.rename(columns={"State": "state"})

df = merge_check(df, plant_summary, on="state", how="left",
                 label="+ plant counts")

In [ ]:
# ── 7. Final cleaning ─────────────────────────────────────────────────────────
KEEP_COLS = [
    "respondent_id", "state", "re_support",
    "ideology", "climate_concern", "education", "income", "age", "gender", "race",
    "coal_share", "n_coal_plants", "n_re_plants"
]

# Keep only columns that exist (handles missing cols gracefully)
keep = [c for c in KEEP_COLS if c in df.columns]
df_final = df[keep].copy()

# Listwise deletion on analytic variables
before = len(df_final)
df_final = df_final.dropna()
after  = len(df_final)
print(f"\nListwise deletion: {before} → {after} rows ({before - after} dropped)")
print(f"\nFinal analytic sample: {df_final.shape}")
print(f"States represented:    {df_final['state'].nunique()}")
print(f"\nDV summary:")
print(df_final["re_support"].describe().round(3))

In [ ]:
# ── 8. Save ───────────────────────────────────────────────────────────────────
out_path = DATA_PROC / "merged_analysis.csv"
df_final.to_csv(out_path, index=False)
print(f"Saved: {out_path}  |  shape: {df_final.shape}")